In [1]:
import pandas  as pd
import numpy as np
import pdb, os, datetime, itertools, time, hashlib
from dotenv import load_dotenv

load_dotenv()
from kdutils.macro2 import base_path
from lumina.genetic.util import create_id

In [2]:
method = 'cicso0'
task_id = '1000301201'

In [3]:
def create_params(params):
    m = hashlib.md5()
    # params可能是字典类型，需要转换为字符串
    if isinstance(params, dict):
        # 将字典按键排序后转换为字符串，确保相同参数组合产生相同hash
        params_str = str(sorted(params.items()))
    else:
        params_str = str(params)
    m.update(bytes(params_str, encoding='UTF-8'))
    return create_id(original=m.hexdigest(), digit=16)

In [4]:
def screening(method, task_id, threshold_ic, threshold_turnover):
    output_dirs = os.path.join(base_path, method, 'evaluate', str(task_id))
    basic_csv = pd.read_csv(os.path.join(output_dirs, "basic", "summary.csv"),
                            index_col=0)
    basic_csv['category'] = 'basic'
    
    derivative_csv = pd.read_csv(os.path.join(output_dirs, "derivative",
                                              "summary.csv"),
                                 index_col=0)
    derivative_csv['category'] = 'derivative'
    results = pd.concat([basic_csv, derivative_csv], axis=0)
    results['abs_ic'] = np.fabs(results['ic'])
    results = results.sort_values(by=['abs_ic'],ascending=False).dropna()
    results = results[(results['abs_ic'] > threshold_ic) & (results['abs_ic'] < 0.5) & (results['turnover'] < threshold_turnover)]
    results['factor_id'] = results['name'].apply(lambda x: create_params(x))
    return results

In [5]:
threshold_ic = 0.02
threshold_turnover = 0.5
results = screening(method, task_id, threshold_ic, threshold_turnover)

In [6]:
results = results[['name','factor_id','category','abs_ic','icir','turnover']]
results.head()

,name,factor_id,category,abs_ic,icir,turnover
276,"EMA(14, MKURT(10, SIGN('cj010_5_10_1')))",1001184837985357,derivative,0.092036,-0.100266,0.004641
479,"MSUM(16, MSKEW(10, SIGN('close')))",1094724787492254,derivative,0.058953,-0.067991,0.007356
125,LAST('cj009_5_10_1'),1091177481111081,basic,0.031336,0.163033,0.068541
124,LAST('cj009_5_10_0'),1063822867992530,basic,0.030343,0.160038,0.072672
123,LAST('cj009_2_3_1'),1059706178353505,basic,0.028959,0.165568,0.232369


In [7]:
#results.apply(lambda x: os.path.join(base_path, 'evaluate', str(task_id), x['category'], 'plot', f"{x['factor_id']}.png"))

results['path'] = (base_path + '/' + str(method) + '/' + 'evaluate' + '/' + str(task_id) + '/' + results['category'].astype(str) \
    + '/' + 'plot' + '/' + results['factor_id'].astype(str) + '.png')
results.head()

,name,factor_id,category,abs_ic,icir,turnover,path
276,"EMA(14, MKURT(10, SIGN('cj010_5_10_1')))",1001184837985357,derivative,0.092036,-0.100266,0.004641,./records/cicso0/evaluate/1000301201/derivativ...
479,"MSUM(16, MSKEW(10, SIGN('close')))",1094724787492254,derivative,0.058953,-0.067991,0.007356,./records/cicso0/evaluate/1000301201/derivativ...
125,LAST('cj009_5_10_1'),1091177481111081,basic,0.031336,0.163033,0.068541,./records/cicso0/evaluate/1000301201/basic/plo...
124,LAST('cj009_5_10_0'),1063822867992530,basic,0.030343,0.160038,0.072672,./records/cicso0/evaluate/1000301201/basic/plo...
123,LAST('cj009_2_3_1'),1059706178353505,basic,0.028959,0.165568,0.232369,./records/cicso0/evaluate/1000301201/basic/plo...


In [8]:
results = results.drop(['factor_id'],axis=1)
results.head()

,name,category,abs_ic,icir,turnover,path
276,"EMA(14, MKURT(10, SIGN('cj010_5_10_1')))",derivative,0.092036,-0.100266,0.004641,./records/cicso0/evaluate/1000301201/derivativ...
479,"MSUM(16, MSKEW(10, SIGN('close')))",derivative,0.058953,-0.067991,0.007356,./records/cicso0/evaluate/1000301201/derivativ...
125,LAST('cj009_5_10_1'),basic,0.031336,0.163033,0.068541,./records/cicso0/evaluate/1000301201/basic/plo...
124,LAST('cj009_5_10_0'),basic,0.030343,0.160038,0.072672,./records/cicso0/evaluate/1000301201/basic/plo...
123,LAST('cj009_2_3_1'),basic,0.028959,0.165568,0.232369,./records/cicso0/evaluate/1000301201/basic/plo...


In [9]:
def make_clickable(val):
    return f'<a target="_blank" href="{val}">{val}</a>'

In [10]:
results['path'] = results['path'].apply(make_clickable)

In [11]:
from IPython.display import display, HTML

In [12]:
display(HTML(results.to_html(escape=False)))

,name,category,abs_ic,icir,turnover,path
276,"EMA(14, MKURT(10, SIGN('cj010_5_10_1')))",derivative,0.092036,-0.100266,0.004641,./records/cicso0/evaluate/1000301201/derivative/plot/1001184837985357.png
479,"MSUM(16, MSKEW(10, SIGN('close')))",derivative,0.058953,-0.067991,0.007356,./records/cicso0/evaluate/1000301201/derivative/plot/1094724787492254.png
125,LAST('cj009_5_10_1'),basic,0.031336,0.163033,0.068541,./records/cicso0/evaluate/1000301201/basic/plot/1091177481111081.png
124,LAST('cj009_5_10_0'),basic,0.030343,0.160038,0.072672,./records/cicso0/evaluate/1000301201/basic/plot/1063822867992530.png
123,LAST('cj009_2_3_1'),basic,0.028959,0.165568,0.232369,./records/cicso0/evaluate/1000301201/basic/plot/1059706178353505.png
159,LAST('count'),basic,0.028102,0.223961,0.174269,./records/cicso0/evaluate/1000301201/basic/plot/1007739925298754.png
126,LAST('cj009_2_3_0'),basic,0.027862,0.162640,0.238293,./records/cicso0/evaluate/1000301201/basic/plot/1040075895161194.png
161,LAST('buy_value'),basic,0.026184,0.213636,0.202486,./records/cicso0/evaluate/1000301201/basic/plot/1010654071100472.png
101,LAST('db006_5_10_1'),basic,0.026049,0.239308,0.104052,./records/cicso0/evaluate/1000301201/basic/plot/1081542555973610.png
46,LAST('ixy002_2_3_1'),basic,0.025817,0.242098,0.274033,./records/cicso0/evaluate/1000301201/basic/plot/1025352503753302.png
